# 🚀 Deploy ke Hugging Face Space
### Step-by-step dari Colab ke Space publik

Notebook ini mengunggah semua file yang dibutuhkan ke Hugging Face Space
sehingga dashboard bisa diakses siapapun via URL publik.

**Prasyarat sebelum menjalankan:**
- ✅ Sudah selesai Notebook 01 (preprocessing) — punya `corpus_structured.json` & `embeddings_labse.npy`
- ✅ Sudah selesai Notebook 02 (training) — punya `legal_comparator.py`
- ✅ Punya akun Hugging Face + token (scope: `write`)
---

## 📦 CELL 1 — Install huggingface_hub

In [ ]:
!pip install -q huggingface_hub gradio
print('✅ Done')

## 🔑 CELL 2 — Login ke Hugging Face

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Ambil token dari Colab Secrets
# Cara set: klik ikon 🔑 di sidebar kiri Colab
# → New secret → Name: HF_TOKEN → Value: token dari huggingface.co/settings/tokens
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('✅ Token ditemukan dari Colab Secrets')
except:
    HF_TOKEN = input('Paste Hugging Face token (scope: write): ')

login(token=HF_TOKEN)
print('✅ Login berhasil!')

## ⚙️ CELL 3 — Konfigurasi Space

In [ ]:
# ── EDIT BAGIAN INI ──────────────────────────────────
HF_USERNAME  = input('Username Hugging Face Anda: ')

MODEL_REPO   = f'{HF_USERNAME}/indonesian-legal-comparator'  # repo model
SPACE_REPO   = f'{HF_USERNAME}/indonesian-legal-comparator-demo'  # repo space
# ─────────────────────────────────────────────────────

print(f'\n📋 Konfigurasi:')
print(f'   Model repo : https://huggingface.co/{MODEL_REPO}')
print(f'   Space URL  : https://huggingface.co/spaces/{SPACE_REPO}')

## 📤 CELL 4 — Upload File Model ke HF Hub

In [ ]:
from huggingface_hub import HfApi
from google.colab import files
import os, shutil, json

api = HfApi()

# Buat model repo
api.create_repo(repo_id=MODEL_REPO, repo_type='model',
                private=False, exist_ok=True)
print(f'✅ Model repo siap: {MODEL_REPO}')

# Upload file model
print('\n📂 Upload file dari komputer:')
print('   - corpus_structured.json')
print('   - embeddings_labse.npy')
print('   - legal_comparator.py')
uploaded = files.upload()

# Siapkan struktur folder
os.makedirs('model_pkg/src', exist_ok=True)
for fname in uploaded:
    if fname == 'legal_comparator.py':
        shutil.copy(fname, 'model_pkg/src/legal_comparator.py')
    else:
        shutil.copy(fname, f'model_pkg/{fname}')

# Buat config.json
config = {
    'model_name'      : 'indonesian-legal-comparator',
    'version'         : '1.0.0',
    'embedding_model' : 'sentence-transformers/LaBSE',
    'threshold_tinggi': 0.80,
    'threshold_sedang': 0.65,
    'languages'       : ['id', 'en'],
    'space_url'       : f'https://huggingface.co/spaces/{SPACE_REPO}'
}
with open('model_pkg/config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Upload ke HF Hub
print('\n📤 Mengupload ke Hugging Face Hub...')
api.upload_folder(
    folder_path    = 'model_pkg',
    repo_id        = MODEL_REPO,
    repo_type      = 'model',
    commit_message = 'Upload Indonesian Legal Comparator v1.0.0'
)
print(f'\n✅ Model berhasil diupload!')
print(f'   🔗 https://huggingface.co/{MODEL_REPO}')

## 🌐 CELL 5 — Buat & Deploy Hugging Face Space

In [ ]:
import os

# Buat Space repo
api.create_repo(
    repo_id  = SPACE_REPO,
    repo_type= 'space',
    space_sdk= 'gradio',
    private  = False,
    exist_ok = True
)
print(f'✅ Space repo siap: {SPACE_REPO}')

# Set environment variable MODEL_REPO_ID di Space
api.add_space_variable(
    repo_id  = SPACE_REPO,
    key      = 'MODEL_REPO_ID',
    value    = MODEL_REPO
)
print(f'✅ Environment variable MODEL_REPO_ID = {MODEL_REPO}')

# Buat folder space
os.makedirs('space_pkg/src', exist_ok=True)

# Download file space yang sudah disiapkan
print('\n📂 Upload file Space:')
print('   - app.py')
print('   - requirements.txt')
print('   - README.md  (Space card)')
space_files = files.upload()

for fname in space_files:
    with open(f'space_pkg/{fname}', 'wb') as f:
        f.write(space_files[fname])

# Salin legal_comparator.py ke src/
shutil.copy('legal_comparator.py', 'space_pkg/src/legal_comparator.py')

# Upload semua ke Space
print('\n📤 Mengupload ke Hugging Face Space...')
api.upload_folder(
    folder_path    = 'space_pkg',
    repo_id        = SPACE_REPO,
    repo_type      = 'space',
    commit_message = 'Deploy Indonesian Legal Comparator Dashboard v1.0.0'
)

print(f'\n🎉 Space berhasil di-deploy!')
print(f'   🔗 Demo URL: https://huggingface.co/spaces/{SPACE_REPO}')
print(f'   ⏳ Tunggu 2-3 menit untuk Space building selesai')
print(f'\n   Setelah live, siapapun bisa akses tanpa install apapun!')

## ✅ CELL 6 — Verifikasi Space Berjalan

In [ ]:
import time
from huggingface_hub import SpaceStage

print('⏳ Menunggu Space selesai building...')
print('   (Biasanya 2-5 menit pertama kali)')
print()

for i in range(20):  # cek tiap 30 detik, max 10 menit
    try:
        info  = api.get_space_runtime(repo_id=SPACE_REPO)
        stage = str(info.stage)
        print(f'   [{i*30:>4}s] Status: {stage}')

        if 'RUNNING' in stage:
            print(f'\n✅ Space RUNNING!')
            print(f'   🌐 https://huggingface.co/spaces/{SPACE_REPO}')
            break
        elif 'ERROR' in stage or 'STOPPED' in stage:
            print(f'\n❌ Space gagal: {stage}')
            print('   Cek logs di HF Space dashboard untuk detail error')
            break
    except Exception as e:
        print(f'   [{i*30:>4}s] Mengecek... ({e})')

    time.sleep(30)

print(f'\n📋 Summary:')
print(f'   Model  : https://huggingface.co/{MODEL_REPO}')
print(f'   Space  : https://huggingface.co/spaces/{SPACE_REPO}')
print(f'\n💡 Share URL ini agar orang lain bisa menggunakan model Anda!')